# Useful Notebook: Report Testing Data Prediction Probabilities and Illustrate Accessing Evaluation Metrics
**This notebook will (1) show users how to access all model evaluation metrics from internal pickle files, and (2) generate model (class 1) prediction probabilities for instances of the respective testing dataset.**

*This notebook is designed to run after having run STREAMLINE (at least phases 1-6) and will use the files from a specific STREAMLINE experiment folder, as well as save new output files to that same folder.*

***
## Notebook Details
STREAMLINE outputs pickled objects with (1) all the metric results, (2) elements needed to build the ROC and PRC plots, as well as (3) the prediction probabilities on the testing data across all datasets, algorithm models, and CV dataset partitions. 

This notebook illustrates how the user can access the pickled metric information saved as a list object. 

It includes (1) grabbing and calculating all average metric scores over the CV partitions, (2) grabbing the elements needed to build the average ROC plot, (3) grabbing the elementes needed to build the average PRC plot, (4) grabbing and reporting average model feature importance scores, and (5) grabbing and reporting the model testing prediction probabilities for each instance of the dataset. 

When run, this last item will generate a new folder (`prediction_probas`) in the pipeline's output experiment folder in the `model_evaluation` folder for each dataset. Here the class 1 prediction probabilities are reported as a `.csv` file for each algorithm and CV partition pair. In these files is the instance's true outcome value, the unique instance ID, and the predicted probability of the instance being class 1 (i.e. which typically encodes cases or the less frequent class). 
 

***
## Notebook Run Parameters
* This notbook has been set up to run 'as-is' on the experiment folder generated when running the demo of STREAMLINE in any mode (if no run parameters were changed). 
* If you have run STREAMLINE on different target data or saved the experiment to some other folder outside of STREAMLINE, you need to edit `experiment_path` below to point to the respective experiment folder.

In [ ]:
experiment_path = "/Users/harshbandhey/Local/Cedars/Urbslab/STREAMLINEv3New/test/out_full_pipeline/DemoExp" # path the target experiment folder 
target_data_list = None # None if user wants to generate output for all analyzed target datasets, otherwise provide a (str) list of target dataset names to run
algorithms = [] # use empty list if user wishes re-evaluate all modeling algorithms that were run in pipeline, otherwise specify a (str) list of algorithm identifiers.

***
## Housekeeping
### Import Packages

In [ ]:
import os
import json
import pandas as pd
import pickle
import numpy as np
from statistics import mean
from scipy import stats
interp = np.interp
import warnings
warnings.filterwarnings('ignore')

# Jupyter Notebook Hack: This code ensures that the results of multiple commands within a given cell are all displayed, rather than just the last. 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

### Automatically Detect Dataset Names

In [ ]:
# Get dataset paths for all completed dataset analyses in experiment folder
experiment_name = experiment_path.split('/')[-1]
remove_list = {
    '.DS_Store', 'metadata.pickle', 'metadata.csv', 'algInfo.pickle',
    'DatasetComparisons', 'jobs', 'jobsCompleted', 'logs', 'KeyFileCopy', 'dask_logs',
    'reporting', 'reporting_replication', 'run_params.pickle', 'runtime',
    experiment_name + '_STREAMLINE_Report.pdf'
}

datasets = []
for d in sorted(os.listdir(experiment_path)):
    dpath = os.path.join(experiment_path, d)
    if d in remove_list or not os.path.isdir(dpath):
        continue
    has_exploratory = os.path.isdir(os.path.join(dpath, 'exploratory'))
    has_model_data = os.path.isdir(os.path.join(dpath, 'model_evaluation')) or os.path.isdir(os.path.join(dpath, 'models'))
    if has_exploratory and has_model_data:
        datasets.append(d)

print("Analyzed Datasets: " + str(datasets))

### Load Other Necessary Parameters

In [ ]:
# Unpickle metadata from previous phase
file = open(experiment_path + '/' + "metadata.pickle", 'rb')
metadata = pickle.load(file)
file.close()
# Load variables specified earlier in the pipeline from metadata
outcome_label = metadata.get('Outcome Label', metadata.get('Class Label', 'Class'))
instance_label = metadata['Instance Label']
cv_partitions = int(metadata['CV Partitions'])

requested_algorithms = list(algorithms) if isinstance(algorithms, list) else []

# Unpickle algorithm information from previous phase
alg_info_path = os.path.join(experiment_path, "algInfo.pickle")
if os.path.exists(alg_info_path):
    with open(alg_info_path, "rb") as file:
        algInfo = pickle.load(file)
else:
    algInfo = {}

algorithms = []
abbrev = {}
for key, value in algInfo.items():
    if isinstance(value, (list, tuple)) and len(value) > 0 and bool(value[0]):
        algorithms.append(key)
        abbrev[key] = value[1] if len(value) > 1 else key

# Fallback: infer algorithms from current output layout
if not algorithms:
    inferred = set()
    scan_datasets = [d for d in datasets if os.path.isdir(os.path.join(experiment_path, d))]
    for ds_name in scan_datasets:
        model_dir = os.path.join(experiment_path, ds_name, "models", "pickledModels")
        if os.path.isdir(model_dir):
            for fname in os.listdir(model_dir):
                if fname.endswith('.pickle') and '_' in fname:
                    inferred.add(fname.rsplit('_', 1)[0])
        metric_dir = os.path.join(experiment_path, ds_name, "model_evaluation", "metrics_by_cv")
        if os.path.isdir(metric_dir):
            for fname in os.listdir(metric_dir):
                if fname.endswith('.json') and '_CV_' in fname:
                    inferred.add(fname.split('_CV_')[0])
    for abr in sorted(inferred):
        algorithms.append(abr)
        abbrev[abr] = abr

if requested_algorithms:
    req = set(requested_algorithms)
    filtered = [a for a in algorithms if a in req or abbrev.get(a) in req]
    if filtered:
        algorithms = filtered

print("Algorithms Ran: " + str(algorithms))

***
## From Pickle: Extract Metric List and Cacluate CV Averages

In [ ]:
def print_results(algorithm, full_path):
    perf_file = full_path + '/model_evaluation/' + abbrev[algorithm] + '_performance.csv'
    if not os.path.exists(perf_file):
        print({'error': 'Missing performance file', 'file': perf_file})
        return

    perf_df = pd.read_csv(perf_file)
    if perf_df.empty:
        print({'error': 'Performance file is empty', 'file': perf_file})
        return

    rename_map = {
        'F1_Score': 'F1 Score',
        'ROC_AUC': 'ROC AUC',
        'PRC_AUC': 'PRC AUC',
        'PRC_APS': 'PRC APS'
    }
    perf_df = perf_df.rename(columns=rename_map)

    summary = {}
    for col in perf_df.columns:
        try:
            summary[col] = float(np.nanmean(perf_df[col].values.astype(float)))
        except Exception:
            continue

    print(summary)

In [ ]:
if target_data_list not in (None, 'None'):
    selected = set(target_data_list) if isinstance(target_data_list, (list, tuple, set)) else {target_data_list}
    datasets = [d for d in datasets if d in selected]

for each in datasets:
    print("---------------------------------------")
    print("Dataset: "+str(each))
    print("---------------------------------------")
    full_path = experiment_path + '/' + each
    for algorithm in algorithms:
        print("Algorithm: "+str(algorithm))
        print_results(algorithm, full_path)

## From Pickle: Extract list of true and false positive rates for constructing ROC

In [ ]:
if target_data_list not in (None, 'None'):
    selected = set(target_data_list) if isinstance(target_data_list, (list, tuple, set)) else {target_data_list}
    datasets = [d for d in datasets if d in selected]

for each in datasets:
    print("---------------------------------------")
    print("Dataset: "+str(each))
    print("---------------------------------------")
    full_path = experiment_path+ '/' + each
    for algorithm in algorithms:
        print("Algorithm: "+str(algorithm))
        tprs = []
        mean_fpr = np.linspace(0, 1, 100)

        for cv_count in range(0, cv_partitions):
            roc_file = full_path + '/model_evaluation/curves_by_cv/' + abbrev[algorithm] + '_CV_' + str(cv_count) + '_roc.json'
            if not os.path.exists(roc_file):
                continue
            with open(roc_file, 'r') as f:
                payload = json.load(f)
            curve = payload.get('micro', payload.get('binary', payload))
            fpr = np.asarray(curve.get('fpr', []), dtype=float)
            tpr = np.asarray(curve.get('tpr', []), dtype=float)
            if len(fpr) < 2 or len(tpr) < 2:
                continue
            order = np.argsort(fpr)
            fpr = fpr[order]
            tpr = tpr[order]
            fpr_u, idx = np.unique(fpr, return_index=True)
            tpr_u = tpr[idx]
            if len(fpr_u) < 2:
                continue
            tprs.append(interp(mean_fpr, fpr_u, tpr_u))
            tprs[-1][0] = 0.0

        if not tprs:
            print({'error': 'No ROC curves found'})
        else:
            results = {'tprs': np.mean(tprs, axis=0)}
            print(results)

## From Pickle: Extract list of precision and recall values for constructing PRC

In [ ]:
if target_data_list not in (None, 'None'):
    selected = set(target_data_list) if isinstance(target_data_list, (list, tuple, set)) else {target_data_list}
    datasets = [d for d in datasets if d in selected]

for each in datasets:
    print("---------------------------------------")
    print("Dataset: "+str(each))
    print("---------------------------------------")
    full_path = experiment_path + '/' + each
    for algorithm in algorithms:
        print("Algorithm: "+str(algorithm))
        precs = []
        mean_recall = np.linspace(0, 1, 100)

        for cv_count in range(0, cv_partitions):
            prc_file = full_path + '/model_evaluation/curves_by_cv/' + abbrev[algorithm] + '_CV_' + str(cv_count) + '_prc.json'
            if not os.path.exists(prc_file):
                continue
            with open(prc_file, 'r') as f:
                payload = json.load(f)
            curve = payload.get('micro', payload.get('binary', payload))
            prec = np.asarray(curve.get('precision', []), dtype=float)
            recall = np.asarray(curve.get('recall', []), dtype=float)
            if len(prec) < 2 or len(recall) < 2:
                continue
            order = np.argsort(recall)
            recall = recall[order]
            prec = prec[order]
            recall_u, idx = np.unique(recall, return_index=True)
            prec_u = prec[idx]
            if len(recall_u) < 2:
                continue
            precs.append(interp(mean_recall, recall_u, prec_u))

        if not precs:
            print({'error': 'No PRC curves found'})
        else:
            results = {'precs': np.mean(precs, axis=0)}
            print(results)

## From Pickle: Extract Average Model Feature Importance Estimates (Over CVs)

In [ ]:
if target_data_list not in (None, 'None'):
    selected = set(target_data_list) if isinstance(target_data_list, (list, tuple, set)) else {target_data_list}
    datasets = [d for d in datasets if d in selected]
print("Dataset: " + str(datasets))

for each in datasets:
    print("---------------------------------------")
    print("Dataset: "+str(each))
    print("---------------------------------------")
    full_path = experiment_path + '/' + each
    for algorithm in algorithms:
        print("Algorithm: "+str(algorithm))
        fi_file = full_path + '/model_evaluation/feature_importance/' + abbrev[algorithm] + '_FI.csv'
        if not os.path.exists(fi_file):
            print({'error': 'Missing FI file', 'file': fi_file})
            continue
        fi_df = pd.read_csv(fi_file)
        if fi_df.empty:
            print({'error': 'Empty FI file', 'file': fi_file})
            continue
        fi_dict = fi_df.mean(axis=0).to_dict()
        print(fi_dict)

## Extract and Output Testing Data Prediction Probabilities 

In [ ]:
if target_data_list not in (None, 'None'):
    selected = set(target_data_list) if isinstance(target_data_list, (list, tuple, set)) else {target_data_list}
    datasets = [d for d in datasets if d in selected]

for each in datasets:
    print("---------------------------------------")
    print("Dataset: "+str(each))
    print("---------------------------------------")

    full_path = experiment_path + '/' + each

    if not os.path.exists(full_path + '/model_evaluation/prediction_probas'):
        os.mkdir(full_path + '/model_evaluation/prediction_probas')

    for algorithm in algorithms:
        print("Algorithm: "+str(algorithm))

        for cv_count in range(0,cv_partitions):
            print("CV: "+str(cv_count))
            model_file = full_path + '/models/pickledModels/' + abbrev[algorithm] + '_' + str(cv_count) + '.pickle'
            if not os.path.exists(model_file):
                print('Missing model file: ' + model_file)
                continue
            with open(model_file, 'rb') as file:
                model = pickle.load(file)
            if not hasattr(model, 'predict_proba'):
                print('Model has no predict_proba; skipping.')
                continue

            test_data = pd.read_csv(full_path + '/CVDatasets/'+each+'_CV_' + str(cv_count) + '_Test.csv')
            id_cols = [outcome_label]
            if instance_label != 'None' and instance_label in test_data.columns:
                id_cols.append(instance_label)
            feature_df = test_data.drop(columns=id_cols)

            probas_all = model.predict_proba(feature_df.values)
            if len(probas_all.shape) != 2:
                print('Unexpected probability shape; skipping.')
                continue
            class_index = 1 if probas_all.shape[1] > 1 else 0

            summary_cols = [outcome_label]
            if instance_label != 'None' and instance_label in test_data.columns:
                summary_cols.append(instance_label)
            probas_summary = test_data[summary_cols].copy()
            probas_summary['1_prob'] = probas_all[:, class_index]

            file_name = full_path + '/model_evaluation/prediction_probas/' + algorithm + '_CV_'+str(cv_count) + '_class1_probas.csv'
            probas_summary.to_csv(file_name, index=False)